## Music Recommendation
### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: March 18, 2026


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [1]:
# Packages
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
# Build SparkSession
spark = (SparkSession.builder
         .appName("autoscrobbler")
         .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/apps/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/21 22:52:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Load user_artist_data
user_artist_data_file = "/standard/ds7200-apt4c/large_datasets/user_artist_data.txt"

user_artist_full_df = spark.read.csv(user_artist_data_file,
                                sep=' ', 
                                header=False,
                                schema="userid INT, artistid INT, playcount INT")

In [4]:
# Sample user_artist_data
user_artist_sample_df = user_artist_full_df.sample(withReplacement=False,
                                            fraction=0.2,
                                            seed=314)

In [5]:
# Correct and collapse alternate artist aliases
# (I realize this is not required, but I wanted to do it as practice)
artist_alias_file = "artist_alias.txt"

artist_alias_df = spark.read.csv(artist_alias_file,
                                 sep='\t',
                                 header=False,
                                 schema="badid INT, goodid INT").dropna()

# Join dataframes
joined_df = user_artist_sample_df.join(
    artist_alias_df,
    on = user_artist_sample_df["artistid"] == artist_alias_df["badid"],
    how = "left")

# Replace incorrect artist aliases with goodid
aliased_df = joined_df.withColumn(
    "artistid",
    F.when(F.col("badid").isNotNull(), F.col("goodid"))
     .otherwise(F.col("artistid")))

# Sum user playcounts by artist
clean_df = (aliased_df
            .groupBy("userid", "artistid")
            .agg(F.sum("playcount").alias("playcount")))

# Count distinct artists per user and filter
count_df = (clean_df
            .groupBy("userid")
            .agg(F.countDistinct("artistid").alias("n_artists")))

filtered_df = (clean_df.join(
    count_df,
    on = "userid",
    how = "inner")
    .filter(F.col("n_artists") >= 10)
    .drop("n_artists"))

sorted_df = filtered_df.orderBy("userid", "artistid").cache()

In [6]:
# Split sample
train, validation, test = sorted_df.randomSplit([0.6, 0.2, 0.2], seed=314)

train.cache()
validation.cache()
test.cache()

print(train.count(), validation.count(), test.count())

sorted_df.unpersist()

2806453 934944 936050


DataFrame[userid: int, artistid: int, playcount: bigint]

In [7]:
# TASK 1 (1 PT)
# Show the first row from the training set
train.show(1)

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|    90|     121|        1|
+------+--------+---------+
only showing top 1 row


In [8]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as Test User 1.
test.show(1)

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|    90|    1495|       60|
+------+--------+---------+
only showing top 1 row


In [9]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01


In [10]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for Test User 1


In [11]:
# TASK 4 (1 PT)
# Briefly explain Recall@K in the context of recommender systems


In [12]:
# TASK 5 (1 PT)
# Calculate Recall@K for Test User 1


In [13]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam, alpha) to maximize Recall@K


In [14]:
# TASK 6 (2 PTS)
# Report the results including best values for (rank, maxIter, regParam, alpha) along with the Recall@K from the validation set


In [15]:
# Retrain the best configuration on train + validation data.


In [16]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.


In [17]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?
